# Fast octopus clip extraction — Colab (A100) edition

Same pipeline as `src/extract_octopus_clips_fast.py`, wired for Colab: **single decode
pass** (octopus CLIP + motion together), **parallel videos** (`WORKERS`), **CUDA CLIP**,
and optional **NVDEC** GPU decode. Reads/writes the **same JSONs + schema + resume**
as the local pipeline, but on a **Drive work folder** so it survives Colab session limits
(re-run to continue where it stopped).

> Runtime → A100 GPU. Put `clip_mlp_hardneg_v2.pt` on Drive; enter server creds when prompted.

## 1. Install

In [ ]:
!pip -q install -U openai-clip
!apt-get -qq install -y xz-utils >/dev/null

# NVDEC-capable ffmpeg (Colab's stock apt ffmpeg has no cuvid). BtbN static GPL build:
!wget -q https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz
!tar xf ffmpeg-master-latest-linux64-gpl.tar.xz
!cp ffmpeg-master-latest-linux64-gpl/bin/ffmpeg  /usr/local/bin/ffmpeg
!cp ffmpeg-master-latest-linux64-gpl/bin/ffprobe /usr/local/bin/ffprobe
!hash -r

# verify NVDEC: expect 'cuda' + h264_cuvid/hevc_cuvid -> then set HWACCEL=True in the config cell
import subprocess
hw = subprocess.run(['ffmpeg','-hwaccels'],capture_output=True,text=True).stdout
dec = subprocess.run(['ffmpeg','-decoders'],capture_output=True,text=True).stdout
HAS_NVDEC = ('cuda' in hw) and ('h264_cuvid' in dec)
print('NVDEC available:', HAS_NVDEC, '(set HWACCEL=True below if True)')
import torch; print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")


## 2. Config

In [ ]:
from pathlib import Path
import os, getpass

# --- Drive work folder (state persists here; resumable across sessions) ---
from google.colab import drive; drive.mount("/content/drive")
DRIVE   = Path("/content/drive/MyDrive/GSOC-Catrobat")
WORK    = DRIVE / "octo-extract"; WORK.mkdir(parents=True, exist_ok=True)
CLIPS_DIR  = WORK / "octopus_clips_verified"; CLIPS_DIR.mkdir(exist_ok=True)
INDEX_JSON = WORK / "octopus_clips_verified.json"
PROCESSED  = WORK / "octopus_clips_processed.json"
CKPT_PATH  = WORK / "clip_mlp_hardneg_v2.pt"     # upload this to the WORK folder on Drive

# --- server creds (entered at runtime, never stored in the notebook) ---
USER = os.environ.get("OCTOPUS_USER") or "octopus"
PASS = os.environ.get("OCTOPUS_PASS") or getpass.getpass("Octopus server password: ")

# --- run knobs ---
WORKERS   = 12          # parallel videos
HWACCEL   = globals().get('HAS_NVDEC', False)   # auto-on if the install cell found NVDEC
LIMIT     = None        # cap videos this run (None = all unprocessed)
DATE      = None        # e.g. "2026-02-22" to restrict
CAMERAS   = ["Right Back", "Right Front", "Right Top"]   # den angles (dropped noisy Right_Left/Right)

# --- gates (same as the local pipeline) ---
SAMPLE_FPS, CLIP_LEN = 1.0, 20
MIN_VISIBLE_FRAC, VIS_THRESH = 0.50, 0.60
MOTION_THRESH, MOTION_PIX = 0.008, 25
DW, DH, BATCH = 640, 360, 64
BASE = "https://repo.octopus-intelligence.org/public/O-vulgaris-Nity-2026-2-20--"
assert CKPT_PATH.exists(), f"upload clip_mlp_hardneg_v2.pt to {WORK}"

## 3. Pipeline (single-pass, parallel) — same logic as extract_octopus_clips_fast.py

In [ ]:
import subprocess, re, json, time, threading, urllib.parse, datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np, torch, torch.nn as nn
from PIL import Image
try:
    import pkg_resources, packaging, packaging.version, packaging.specifiers, packaging.requirements
    pkg_resources.packaging = packaging
except Exception: pass
import clip as clip_lib

device = "cuda"
_model_lock, _json_lock = threading.Lock(), threading.Lock()

def auth(u): return u.replace("https://", f"https://{USER}:{PASS}@")
def letterbox(img, size=224, fill=(128,128,128)):
    w,h=img.size; s=size/max(w,h); nw,nh=max(1,round(w*s)),max(1,round(h*s))
    img=img.resize((nw,nh),Image.BICUBIC); cv=Image.new("RGB",(size,size),fill)
    cv.paste(img,((size-nw)//2,(size-nh)//2)); return cv

def _curl(u): return subprocess.run(["curl","-s","--user",f"{USER}:{PASS}",u],capture_output=True,text=True).stdout
def list_dates(): return sorted(set(re.findall(r'href="(\d{4}-\d{2}-\d{2})/"', _curl(f"{BASE}/Right%20Top/Local/"))))
def list_segments(cam,date):
    enc=urllib.parse.quote(cam); out=_curl(f"{BASE}/{enc}/Local/{date}/"); rows=[]
    for f in re.findall(r'href="([^"]+\.mp4)"', out):
        m=re.match(r"(\d+)--", f)
        if not m: continue
        seg=m.group(1); cu=cam.replace(" ","_")
        rows.append({"video":f"data/aquarium/full/{date}/{seg}/{cu}.mp4","date":date,"segment":seg,"camera":cu,
                     "url":f"{BASE}/{enc}/Local/{date}/{f}"})
    return rows
def enumerate_candidates(dates,cams):
    tasks=[(c,d) for d in dates for c in cams]; out=[]
    with ThreadPoolExecutor(max_workers=16) as ex:
        for r in ex.map(lambda a:list_segments(*a),tasks): out.extend(r)
    return out

def load_json(p,default): return json.load(open(p)) if p.exists() else default
def init_reg():
    proc=load_json(PROCESSED,{"task":"octopus_clip_extraction","description":"do not reprocess","updated_at":None,"count":0,"processed":[]})
    idx=load_json(INDEX_JSON,{"description":"Extracted 20s octopus clips.","model":CKPT_PATH.name,"updated_at":None,"count":0,"clips":[]})
    return proc,idx
def save_reg(proc,idx):
    now=datetime.datetime.now().isoformat(timespec="seconds")
    proc["count"]=len(proc["processed"]); proc["updated_at"]=now
    idx["count"]=len(idx["clips"]); idx["updated_at"]=now
    json.dump(proc,open(PROCESSED,"w"),indent=2); json.dump(idx,open(INDEX_JSON,"w"),indent=2)

ck=torch.load(CKPT_PATH,map_location=device); CM,PP=clip_lib.load(ck["clip_model"],device=device); CM.eval()
def _clf(ck):
    feat=ck["feat_dim"]; hid=[int(x) for x in ck["arch"].replace("mlp_","").split("_")]; dims=[feat]+hid+[2]; L=[]
    for i in range(len(dims)-1):
        L.append(nn.Linear(dims[i],dims[i+1]))
        if i<len(dims)-2: L+=[nn.ReLU(),nn.Dropout(0.3)]
    return nn.Sequential(*L)
CLF=_clf(ck).to(device); CLF.load_state_dict(ck["state_dict"]); CLF.eval(); VIS=ck.get("label_map",{}).get("visible",1)
print("model ready:",ck["clip_model"],ck.get("arch"),f"acc={ck.get('test_acc',0):.1%}")

def scan_video(url):
    cmd=["ffmpeg"]+(["-hwaccel","cuda"] if HWACCEL else [])+["-loglevel","error","-i",auth(url),
        "-vf",f"fps={SAMPLE_FPS},scale={DW}:{DH}","-f","image2pipe","-vcodec","rawvideo","-pix_fmt","rgb24","-"]
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL); fsize=DW*DH*3
    pv,motion,buf=[],[],[]; prev=None
    def flush():
        if not buf: return
        with _model_lock:
            b=torch.stack([PP(letterbox(im)) for im in buf]).to(device)
            with torch.no_grad():
                f=CM.encode_image(b).float(); f=f/f.norm(dim=-1,keepdim=True)
                p=torch.softmax(CLF(f),1)[:,VIS]
        pv.extend(p.cpu().tolist()); buf.clear()
    while True:
        raw=proc.stdout.read(fsize)
        if len(raw)<fsize: break
        arr=np.frombuffer(raw,np.uint8).reshape(DH,DW,3); g=arr.astype(np.float32).mean(axis=2)
        if prev is not None:
            d=np.abs(g-prev); d[int(DH*0.88):,int(DW*0.60):]=0.0; motion.append(float((d>MOTION_PIX).mean()))
        prev=g; buf.append(Image.fromarray(arr))
        if len(buf)>=BATCH: flush()
    flush(); proc.stdout.close(); proc.wait()
    return np.array(pv,np.float32), np.array(motion,np.float32)

def find_windows(pv,mot):
    L=int(CLIP_LEN*SAMPLE_FPS); N=len(pv); m=np.zeros(N,np.float32); m[1:1+len(mot)]=mot[:max(0,N-1)]
    out,s=[],0
    while s+L<=N:
        vf=float((pv[s:s+L]>=VIS_THRESH).mean()); mm=float(m[s:s+L].mean())
        if vf>MIN_VISIBLE_FRAC and mm>=MOTION_THRESH:
            out.append({"start_sec":s,"end_sec":s+L,"visible_frac":round(vf,3),"mean_motion":round(mm,5)}); s+=L
        else: s+=1
    return out
def hhmmss(x): return f"{x//60:02d}:{x%60:02d}"
def extract_clip(url,s,e,path):
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and path.stat().st_size>10000: return True
    subprocess.run(["ffmpeg","-loglevel","error","-y","-ss",str(s),"-to",str(e),"-i",auth(url),"-c","copy",str(path)],capture_output=True)
    return path.exists() and path.stat().st_size>10000

def process_video(c):
    try: pv,mot=scan_video(c["url"])
    except Exception as e: print("  ! scan failed",c["segment"],c["camera"],e,flush=True); return c,[],0
    if len(pv)==0: return c,[],0
    entries=[]
    for w in find_windows(pv,mot):
        path=CLIPS_DIR/c["date"]/c["segment"]/f"{c['camera']}_{w['start_sec']:04d}-{w['end_sec']:04d}.mp4"
        if extract_clip(c["url"],w["start_sec"],w["end_sec"],path):
            entries.append({"video":c["video"],"video_url":c["url"],"date":c["date"],"segment":c["segment"],
                "camera":c["camera"],"start_sec":w["start_sec"],"end_sec":w["end_sec"],
                "video_timeline":f"{hhmmss(w['start_sec'])}-{hhmmss(w['end_sec'])}",
                "visible_frac":w["visible_frac"],"mean_motion":w["mean_motion"],
                "clip_path":str(path.relative_to(WORK)),"added_at":datetime.datetime.now().isoformat(timespec="seconds")})
    return c,entries,int(len(pv))
print("pipeline ready.")

## 4. Run (parallel, resumable — re-run this cell to continue)

In [ ]:
proc_reg,clip_idx=init_reg()
done={r["video"] for r in proc_reg["processed"]}
dates=[DATE] if DATE else list_dates()
cands=enumerate_candidates(dates,CAMERAS)
todo=[c for c in cands if c["video"] not in done]
if LIMIT: todo=todo[:LIMIT]
print(f"{len(cands)} candidates; {len(todo)} to process | {WORKERS} workers | hwaccel={HWACCEL}\n"+"-"*60,flush=True)

t0=time.perf_counter(); n=0
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs={ex.submit(process_video,c):c for c in todo}
    for fut in as_completed(futs):
        c,entries,nf=fut.result()
        with _json_lock:
            clip_idx["clips"].extend(entries)
            proc_reg["processed"].append({"video":c["video"],"date":c["date"],"segment":c["segment"],
                "camera":c["camera"],"n_clips":len(entries),"n_frames":nf,"sources":["extract_fast_colab"]})
            done.add(c["video"]); save_reg(proc_reg,clip_idx)
        n+=1; rate=n/max(1e-9,time.perf_counter()-t0)
        print(f"  [{n}/{len(todo)}] {c['date']} {c['segment']} {c['camera']} -> {len(entries)} clips | {rate*60:.1f} vids/min",flush=True)
print("-"*60+f"\nDONE. clips: {clip_idx['count']} | processed videos: {proc_reg['count']}")

## Notes
- **Resumable**: state lives in the Drive `octo-extract/` folder (`octopus_clips_verified.json` +
  ledger + clips). If Colab disconnects, just re-run the Run cell — it skips processed videos.
- **Speed**: raise `WORKERS` if the A100/network allow; set `HWACCEL=True` only if your ffmpeg has NVDEC.
- Clips + index are already on Drive — copy the JSON back into `src/` (or merge) when done.